# 🌍 Job Scout — Camila
### Lanceur — profil Camila

Ce notebook est un **lanceur**. Toute la logique vit dans le package `jobscout/`
et le profil dans `profiles/camila.json` (non versionné).

- Modifier les mots-clés, les critères de scoring ou les sources : éditer le profil,
  puis `python3 tools/build_profiles.py` si on repart des notebooks d'origine.
- Comprendre l'architecture : voir `ARCHITECTURE.md`.

**Rien n'est dépensé tant que la cellule 2 n'a pas été confirmée.**


In [ ]:
%pip install -q anthropic openpyxl httpx feedparser beautifulsoup4 apify-client pydantic

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "jobscout").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from jobscout.connectors import Secrets
from jobscout.pipeline import RunOptions, run
from jobscout.profile import load_profile
from jobscout.store.json_store import JsonStore

PROFILE = load_profile(REPO / "profiles" / "camila.json", base_dir=REPO)
STORE   = JsonStore(REPO / "store_data")
SECRETS = Secrets.from_env()

print(f"Profil   : {PROFILE.display_name} ({PROFILE.profile_id})")
print(f"Sources  : {', '.join(s.type for s in PROFILE.enabled_sources())}")
print(f"ANTHROPIC: {'OK' if SECRETS.anthropic_api_key else 'MANQUANT'}")
print(f"APIFY    : {'OK' if SECRETS.apify_token else 'MANQUANT'}")
print(f"Deja vues: {len(STORE.seen_keys(PROFILE.profile_id))} offres en cache")

---
## 1 — Estimation du coût

Aucun appel réseau, aucune dépense. Affiche ce que le run coûterait et
l'empreinte à recopier ci-dessous pour le confirmer.

In [ ]:
from jobscout.collect import plan_run
from cost_guard import check_run_cost

plans, _ = plan_run(PROFILE, REPO / "dumps")
estimate = check_run_cost(plans, PROFILE.cost_policy)

print(f"Décision   : {estimate.decision.value}")
print(f"Coût estimé: ${estimate.usd_total:.2f}   "
      f"(plafond dur ${estimate.policy.max_cost_per_run:.2f}, "
      f"confirmation au-delà de ${estimate.policy.confirm_above:.2f})")
print(f"Démarrages : {estimate.n_starts_total}   "
      f"Résultats (pire cas) : {estimate.worst_case_results_total}")
for c in estimate.per_connector:
    print(f"   [{c.connector}] ${c.usd_total:.4f}")
    for w in c.warnings:
        print(f"      /!\\ {w}")
print(f"\nEmpreinte à confirmer : {estimate.fingerprint}")
print("\nÀ cela s'ajoute le coût Haiku du scoring, ~$0.005 par offre.")

---
## 2 — Lancement

`CONFIRM` : coller l'empreinte affichée ci-dessus pour autoriser un run payant.
Laisser vide tant qu'on ne veut rien dépenser.

`MAX_JOBS` : plafond de sécurité sur le nombre d'offres scorées. `None` = pas de limite.

In [ ]:
CONFIRM  = ""      # <- coller l'empreinte ici pour confirmer
MAX_JOBS = 25      # <- None pour tout scorer

report = run(
    PROFILE, STORE, SECRETS,
    RunOptions(
        confirmed_fingerprint=CONFIRM or None,
        max_jobs_to_score=MAX_JOBS,
        dumps_dir=str(REPO / "dumps"),
        export_xlsx_path=str(REPO / "job_scout_camila/job_scout_camila_results.xlsx"),
    ),
    repo_dir=REPO,
)

print(f"\nStatut : {report.status}")
if not report.ran:
    print(report.reason)
else:
    for k, v in report.counts.items():
        print(f"   {k:20} {v}")
    print(f"   export -> {report.export_path}")

---
## 3 — Relire les résultats

Lit le store, sans rien relancer ni rien dépenser. C'est cette table que
l'application web lira (`job_results`).

In [ ]:
results = sorted(STORE.get_results(PROFILE.profile_id, verdicts=["YES"]),
                 key=lambda r: -r.score)

print(f"{len(results)} offres retenues\n")
for r in results[:10]:
    labels = {d.key: d.label for d in PROFILE.dimensions}
    breakdown = " ".join(f"{labels[k]}:{v}" for k, v in r.breakdown.items() if k in labels)
    print(f"[{r.score:>4}] {r.title[:52]}")
    print(f"        {r.company[:40]}  |  {r.location or '?'}  |  {r.source}")
    print(f"        {breakdown}")
    print(f"        {r.one_liner[:96]}")
    print(f"        {r.url[:90]}\n")